In [1]:
!nvidia-smi

Sat Sep  5 16:45:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys

print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.0 MB/s eta 0:00:00


In [5]:
from ultralytics import YOLO

print("Ultralytics imported successfully")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics imported successfully


In [6]:
!mkdir -p /content/widerface

In [7]:
%cd /content/widerface

/content/widerface


In [8]:
!wget -q --show-progress \
  -O WIDER_train.zip \
  https://huggingface.co/datasets/CUHK-CSE/wider_face/resolve/main/data/WIDER_train.zip

WIDER_train.zip     100%[===================>]   1.36G   139MB/s    in 10s     


In [9]:
!wget -q --show-progress \
  -O WIDER_val.zip \
  https://huggingface.co/datasets/CUHK-CSE/wider_face/resolve/main/data/WIDER_val.zip

WIDER_val.zip       100%[===================>] 345.95M   159MB/s    in 2.2s    


In [10]:
!wget -q --show-progress \
  -O wider_face_split.zip \
  https://huggingface.co/datasets/CUHK-CSE/wider_face/resolve/main/data/wider_face_split.zip

wider_face_split.zi 100%[===================>]   3.42M  10.4MB/s    in 0.3s    


In [11]:
!unzip -q WIDER_train.zip
!unzip -q WIDER_val.zip
!unzip -q wider_face_split.zip

In [12]:
!find WIDER_train/images -type f | wc -l

12880


In [13]:
!find WIDER_val/images -type f | wc -l

3226


In [14]:
from pathlib import Path

DATASET_ROOT = Path("/content/face_dataset")

(DATASET_ROOT / "images/train").mkdir(parents=True, exist_ok=True)
(DATASET_ROOT / "images/val").mkdir(parents=True, exist_ok=True)

(DATASET_ROOT / "labels/train").mkdir(parents=True, exist_ok=True)
(DATASET_ROOT / "labels/val").mkdir(parents=True, exist_ok=True)

In [15]:
from pathlib import Path
from PIL import Image
import shutil


# ============================================================
# Project paths
# ============================================================

WIDER_ROOT = Path("/content/widerface")
DATASET_ROOT = Path("/content/face_dataset")


# ============================================================
# Helper
# ============================================================

def looks_like_image_path(line):
    """
    WIDER FACE image paths look like:

        0--Parade/0_Parade_Parade_0_100.jpg
    """
    return line.lower().endswith((".jpg", ".jpeg", ".png"))


# ============================================================
# Convert one split
# ============================================================

def convert_split(split):

    print()
    print(f"Converting {split} split...")
    print("-" * 50)

    # --------------------------------------------------------
    # Paths
    # --------------------------------------------------------

    if split == "train":
        image_root = WIDER_ROOT / "WIDER_train" / "images"
    else:
        image_root = WIDER_ROOT / "WIDER_val" / "images"

    annotation_file = (
        WIDER_ROOT
        / "wider_face_split"
        / f"wider_face_{split}_bbx_gt.txt"
    )

    output_images = DATASET_ROOT / "images" / split
    output_labels = DATASET_ROOT / "labels" / split

    # Create output directories
    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Check annotation file
    # --------------------------------------------------------

    if not annotation_file.exists():
        print(
            f"[ERROR] Annotation file not found:\n"
            f"        {annotation_file}"
        )
        return

    # --------------------------------------------------------
    # Read annotations
    # --------------------------------------------------------

    try:
        with open(annotation_file, "r") as f:
            # Remove blank lines
            lines = [line.strip() for line in f if line.strip()]
    except Exception as e:
        print(f"[ERROR] Could not read annotation file:")
        print(f"        {annotation_file}")
        print(f"        Error: {e}")
        return

    # --------------------------------------------------------
    # Counters
    # --------------------------------------------------------

    i = 0

    image_count = 0
    skipped_count = 0
    missing_count = 0
    corrupt_count = 0
    malformed_count = 0

    # ========================================================
    # Main parsing loop
    # ========================================================

    while i < len(lines):

        # ----------------------------------------------------
        # Find next image path
        # ----------------------------------------------------

        if not looks_like_image_path(lines[i]):

            print(
                f"[WARNING] Unexpected line at index {i}: "
                f"{lines[i]}"
            )

            malformed_count += 1
            i += 1

            continue

        # ----------------------------------------------------
        # Image path
        # ----------------------------------------------------

        image_relative_path = lines[i]
        i += 1

        # ----------------------------------------------------
        # Make sure face count exists
        # ----------------------------------------------------

        if i >= len(lines):

            print(
                f"[WARNING] Missing face count for: "
                f"{image_relative_path}"
            )

            skipped_count += 1
            break

        # ----------------------------------------------------
        # Number of faces
        # ----------------------------------------------------

        try:

            num_faces = int(lines[i])

        except ValueError:

            print(
                f"[WARNING] Invalid face count for: "
                f"{image_relative_path}"
            )

            print(
                f"          Found: {lines[i]}"
            )

            malformed_count += 1
            skipped_count += 1

            # ------------------------------------------------
            # Parser recovery
            #
            # Search forward until another image path is found.
            # ------------------------------------------------

            while i < len(lines):

                if looks_like_image_path(lines[i]):
                    break

                i += 1

            continue

        i += 1

        # ----------------------------------------------------
        # Construct image path
        # ----------------------------------------------------

        image_path = image_root / image_relative_path

        # ----------------------------------------------------
        # Check image exists
        # ----------------------------------------------------

        if not image_path.exists():

            print(
                f"[WARNING] Image not found: "
                f"{image_relative_path}"
            )

            missing_count += 1
            skipped_count += 1

            # Skip this image's annotation lines
            i += num_faces

            continue

        # ----------------------------------------------------
        # Read image dimensions ONCE
        # ----------------------------------------------------

        try:

            with Image.open(image_path) as image:
                image_width, image_height = image.size

        except Exception as e:

            print(
                f"[WARNING] Could not open image: "
                f"{image_relative_path}"
            )

            print(f"          Error: {e}")

            corrupt_count += 1
            skipped_count += 1

            # Skip annotation lines
            i += num_faces

            continue

        # ----------------------------------------------------
        # Make sure dimensions are valid
        # ----------------------------------------------------

        if image_width <= 0 or image_height <= 0:

            print(
                f"[WARNING] Invalid image dimensions: "
                f"{image_relative_path}"
            )

            print(
                f"          Size: "
                f"{image_width} x {image_height}"
            )

            corrupt_count += 1
            skipped_count += 1

            i += num_faces

            continue

        # ----------------------------------------------------
        # Convert bounding boxes
        # ----------------------------------------------------

        yolo_labels = []

        box_error = False

        for _ in range(num_faces):

            # Safety check
            if i >= len(lines):

                print(
                    f"[WARNING] Unexpected end of annotation "
                    f"while processing {image_relative_path}"
                )

                box_error = True
                break

            values = lines[i].split()
            i += 1

            # ------------------------------------------------
            # WIDER FACE annotation should contain at least:
            #
            # x y width height
            # ------------------------------------------------

            if len(values) < 4:

                print(
                    f"[WARNING] Malformed bounding box in "
                    f"{image_relative_path}: {values}"
                )

                box_error = True
                malformed_count += 1

                continue

            # ------------------------------------------------
            # Parse coordinates
            # ------------------------------------------------

            try:

                x = float(values[0])
                y = float(values[1])
                width = float(values[2])
                height = float(values[3])

            except ValueError:

                print(
                    f"[WARNING] Invalid bounding box in "
                    f"{image_relative_path}: {values}"
                )

                box_error = True
                malformed_count += 1

                continue

            # ------------------------------------------------
            # Ignore invalid boxes
            # ------------------------------------------------

            if width <= 0 or height <= 0:
                continue

            # ------------------------------------------------
            # Convert:
            #
            # x, y, width, height
            #
            # ->
            #
            # x_center, y_center, width, height
            # ------------------------------------------------

            x_center = x + width / 2
            y_center = y + height / 2

            # ------------------------------------------------
            # Normalize to 0-1
            # ------------------------------------------------

            x_center /= image_width
            y_center /= image_height

            width /= image_width
            height /= image_height

            # ------------------------------------------------
            # Clamp values
            # ------------------------------------------------

            x_center = max(0.0, min(1.0, x_center))
            y_center = max(0.0, min(1.0, y_center))

            width = max(0.0, min(1.0, width))
            height = max(0.0, min(1.0, height))

            # ------------------------------------------------
            # Class 0 = face
            # ------------------------------------------------

            yolo_labels.append(
                f"0 {x_center:.6f} {y_center:.6f} "
                f"{width:.6f} {height:.6f}"
            )

        # ----------------------------------------------------
        # If annotation was completely broken, skip image
        # ----------------------------------------------------

        if box_error and not yolo_labels:

            print(
                f"[WARNING] Skipping image because no valid "
                f"bounding boxes were found: "
                f"{image_relative_path}"
            )

            skipped_count += 1
            continue

        # ----------------------------------------------------
        # Skip images with zero valid faces
        # ----------------------------------------------------

        if not yolo_labels:

            print(
                f"[WARNING] No valid bounding boxes for: "
                f"{image_relative_path}"
            )

            skipped_count += 1
            continue

        # ----------------------------------------------------
        # Flatten directory structure
        #
        # 0--Parade/abc.jpg
        #
        # ->
        #
        # 0--Parade_abc.jpg
        # ----------------------------------------------------

        output_name = image_relative_path.replace("/", "_")

        output_image = output_images / output_name

        output_label = output_labels / (
            Path(output_name).stem + ".txt"
        )

        # ----------------------------------------------------
        # Copy image
        # ----------------------------------------------------

        try:

            shutil.copy2(
                image_path,
                output_image
            )

        except Exception as e:

            print(
                f"[WARNING] Failed to copy: "
                f"{image_relative_path}"
            )

            print(f"          Error: {e}")

            skipped_count += 1
            continue

        # ----------------------------------------------------
        # Write YOLO labels
        # ----------------------------------------------------

        try:

            with open(output_label, "w") as f:

                f.write(
                    "\n".join(yolo_labels)
                )

        except Exception as e:

            print(
                f"[WARNING] Failed to write label for: "
                f"{image_relative_path}"
            )

            print(f"          Error: {e}")

            # Remove image if label creation failed
            if output_image.exists():

                try:
                    output_image.unlink()
                except Exception:
                    pass

            skipped_count += 1
            continue

        # ----------------------------------------------------
        # Successfully processed
        # ----------------------------------------------------

        image_count += 1

        if image_count % 1000 == 0:

            print(
                f"Processed {image_count} images "
                f"(skipped: {skipped_count})"
            )

    # ========================================================
    # Split summary
    # ========================================================

    print()
    print(f"{split} conversion complete!")
    print("-" * 50)
    print(f"Images converted : {image_count}")
    print(f"Images skipped   : {skipped_count}")
    print(f"Missing images   : {missing_count}")
    print(f"Corrupt images   : {corrupt_count}")
    print(f"Malformed blocks : {malformed_count}")


# ============================================================
# Main
# ============================================================

convert_split("train")
convert_split("val")

print()
print("================================")
print("Dataset conversion complete!")
print("================================")


Converting train split...
--------------------------------------------------
[WARNING] No valid bounding boxes for: 0--Parade/0_Parade_Parade_0_452.jpg
[WARNING] Unexpected line at index 10423: 0 0 0 0 0 0 0 0 0 0
Processed 1000 images (skipped: 1)
Processed 2000 images (skipped: 1)
Processed 3000 images (skipped: 1)
[WARNING] No valid bounding boxes for: 2--Demonstration/2_Demonstration_Political_Rally_2_444.jpg
[WARNING] Unexpected line at index 86538: 0 0 0 0 0 0 0 0 0 0
Processed 4000 images (skipped: 2)
Processed 5000 images (skipped: 2)
Processed 6000 images (skipped: 2)
Processed 7000 images (skipped: 2)
[WARNING] No valid bounding boxes for: 39--Ice_Skating/39_Ice_Skating_iceskiing_39_380.jpg
[WARNING] Unexpected line at index 133393: 0 0 0 0 0 0 0 0 0 0
Processed 8000 images (skipped: 3)
Processed 9000 images (skipped: 3)
[WARNING] No valid bounding boxes for: 46--Jockey/46_Jockey_Jockey_46_576.jpg
[WARNING] Unexpected line at index 145713: 0 0 0 0 0 0 0 0 0 0
Processed 10000

In [16]:
yaml_content = """
path: /content/face_dataset

train: images/train
val: images/val

names:
  0: face
"""

with open("/content/face_dataset/data.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)


path: /content/face_dataset

train: images/train
val: images/val

names:
  0: face



In [17]:
model = YOLO("/content/yolo26n.pt")

In [18]:
results = model.train(
    data="/content/face_dataset/data.yaml",

    epochs=20,

    imgsz=640,

    batch=16,

    device=0,

    workers=2,

    project="/content/runs",

    name="face_detection"
)

Ultralytics 8.4.140 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/face_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=face_detection, nbs=6

In [19]:
from google.colab import files

files.download("/content/runs/face_detection/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>